# 🍏 Fun & Fit Health Advisor Agent Tutorial 🍎

Welcome to our **Fun & Fit Health Advisor Agent** tutorial, where you'll use **Azure AI Foundry** SDKs to create a playful (yet carefully disclaimed!) health and fitness assistant. We'll:

1. **Initialize** our project using **azure-ai-projects**.
2. **Create an Agent** specialized in providing general wellness and nutritional advice (with disclaimers!).
3. **Manage conversations** about fitness, nutrition, and general health topics.
4. **Showcase logging and tracing** with **OpenTelemetry**.
5. **Demonstrate** how to incorporate tools, safety disclaimers, and basic best practices.

### ⚠️ Important Medical Disclaimer ⚠️
> **The health information provided by this notebook is for general educational and entertainment purposes only and is not intended as a substitute for professional medical advice, diagnosis, or treatment.** Always seek the advice of your physician or other qualified health provider with any questions you may have regarding a medical condition. Never disregard professional medical advice or delay seeking it because of something you read or receive from this notebook.




## 1. Initial Setup
We'll start by importing needed libraries, loading environment variables, and initializing an **AIProjectClient** so we can do all the agent-related actions. Let's do it! 🎉


In [3]:
! pip install "azure-ai-agents>=1.2.0b3"

  Obtaining dependency information for azure-ai-agents>=1.2.0b3 from https://files.pythonhosted.org/packages/96/d0/930c522f5fa9da163de057e57f8b44539424e13f46618c52624ebc712293/azure_ai_agents-1.2.0b6-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.9/217.9 kB 20.6 MB/s eta 0:00:00
  Attempting uninstall: azure-ai-agents
    Found existing installation: azure-ai-agents 1.1.0
    Uninstalling azure-ai-agents-1.1.0:
      Successfully uninstalled azure-ai-agents-1.1.0

[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [1]:
# Import required libraries
import os  # For environment variables and path operations
import time  # For adding delays if needed
from pathlib import Path  # For cross-platform path handling
import json


# Import Azure and utility libraries
from azure.identity import DefaultAzureCredential  # For Azure authentication
from azure.ai.projects import AIProjectClient  # Main client for AI Projects

def find_cred_json(start_path):
    current = Path(start_path)
    while current != current.parent:
        cred_file = current / 'cred.json'
        if cred_file.exists():
            return str(cred_file)
        current = current.parent
    return None

start_dir = os.getcwd()  # Use current directory
file_path = find_cred_json(start_dir)

print(f"Found cred.json at: {file_path}")

try:
    with open(file_path, 'r') as f:
        loaded_config = json.load(f)

    print("Project Endpoint:", loaded_config['PROJECT_ENDPOINT'])
    print("Model Deployment ID:", loaded_config['MODEL_DEPLOYMENT_NAME'])

    project_client = AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint=loaded_config["PROJECT_ENDPOINT"],
    )

    print("✅ Successfully initialized AIProjectClient")

except FileNotFoundError:
    print(f"❌ Could not find file at: {file_path}")
except json.JSONDecodeError:
    print("❌ File exists but contains invalid JSON")
except TypeError:
    print("❌ File path was None — cred.json not found in any parent directories.")


Found cred.json at: /workspaces/Azure-ai-foundry01/cred.json
Project Endpoint: https://sarath-6690-resource.services.ai.azure.com/api/projects/sarath-6690
Model Deployment ID: gpt-4o
✅ Successfully initialized AIProjectClient


## 2. Creating our Fun & Fit Health Advisor Agent 🏋️

We'll create an Agent specialized in general health and wellness. We'll explicitly mention disclaimers in its instructions, so it never forgets to keep it safe! The instructions also ask the agent to focus on general fitness, dietary tips, and always encourage the user to seek professional advice.


In [5]:
import os
import json
from pathlib import Path
from azure.identity import DefaultAzureCredential
from azure.ai.agents import AgentsClient  # ✅ Use this directly

# Your existing cred.json loading stays the same
# ...

# Replace project_client.agents with a standalone AgentsClient
agents_client = AgentsClient(
    endpoint=loaded_config["PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential()
)

def create_health_advisor_agent():
    try:
        model_name = loaded_config.get("MODEL_DEPLOYMENT_NAME", "gpt-4o")

        agent = agents_client.create_agent(   # ✅ create_agent lives here
            model=model_name,
            name="fun-fit-health-advisor",
            instructions="""
            You are a friendly AI Health Advisor.
            You provide general health, fitness, and nutrition information, but always:
            1. Include medical disclaimers.
            2. Encourage the user to consult healthcare professionals.
            3. Provide general, non-diagnostic advice around wellness, diet, and fitness.
            4. Clearly remind them you're not a doctor.
            5. Encourage safe and balanced approaches to exercise and nutrition.
            """
        )
        print(f"🎉 Created health advisor agent, ID: {agent.id}")
        return agent
    except Exception as e:
        print(f"❌ Error creating agent: {str(e)}")
        return None

health_advisor = create_health_advisor_agent()

🎉 Created health advisor agent, ID: asst_IfPYrS6ysVPcCx3e6iLJZN0X


## 3. Managing Our Health Conversations 💬
A conversation (or *thread*) is where we'll store the user's messages and the agent's responses about health topics. Let's create a new thread dedicated to Health & Fitness Q&A.


In [7]:
def start_health_conversation():
    try:
        thread = agents_client.threads.create()  # ✅ use agents_client
        print(f"📝 Created a new conversation thread, ID: {thread.id}")
        return thread
    except Exception as e:
        print(f"❌ Error creating thread: {e}")
        return None

health_thread = start_health_conversation()

📝 Created a new conversation thread, ID: thread_2vXtONgJZontqzNO7ekOlDnF


## 4. Asking Health & Fitness Questions 🏃‍♂️
We'll create messages from the user about typical health questions. For example, **"How do I calculate my BMI?"** or **"What's a balanced meal for an active lifestyle?"**. We'll let our Health Advisor Agent respond, always remembering that disclaimer!


In [10]:
def ask_health_question(thread_id, user_question):
    try:
        return agents_client.messages.create(
            thread_id=thread_id,
            role="user",
            content=user_question
        )
    except Exception as e:
        print(f"❌ Error adding user message: {e}")
        return None

def process_thread_run(thread_id, agent_id):
    try:
        run = agents_client.runs.create(
            thread_id=thread_id,
            agent_id=agent_id
        )
        while run.status in ["queued", "in_progress", "requires_action"]:
            time.sleep(1)
            run = agents_client.runs.get(
                thread_id=thread_id,
                run_id=run.id
            )
        print(f"🤖 Run completed with status: {run.status}")
        return run
    except Exception as e:
        print(f"❌ Error processing thread run: {e}")
        return None

def view_thread_messages(thread_id):
    try:
        messages = agents_client.messages.list(thread_id=thread_id)
        print("\n🗣️ Conversation so far (oldest to newest):")
        for m in messages:
            print(f"{m.role.upper()}: {m.content}\n")
        print("-----------------------------------\n")
    except Exception as e:
        print(f"❌ Error viewing thread: {e}")

### Example Queries
Let's do some quick queries now to see the agent's disclaimers and how it handles typical health questions. We'll ask about **BMI** and about **balanced meal** for an active lifestyle.


In [11]:
if health_advisor and health_thread:
    ask_health_question(health_thread.id, "How do I calculate my BMI, and what does it mean?")
    process_thread_run(health_thread.id, health_advisor.id)
    ask_health_question(health_thread.id, "Can you give me a balanced meal plan for someone who exercises 3x a week?")
    process_thread_run(health_thread.id, health_advisor.id)
    view_thread_messages(health_thread.id)
else:
    print("❌ Setup incomplete – agent or thread is missing.")


🤖 Run completed with status: RunStatus.COMPLETED
🤖 Run completed with status: RunStatus.COMPLETED

🗣️ Conversation so far (oldest to newest):
ASSISTANT: [{'type': 'text', 'text': {'value': "Of course! A balanced meal plan for someone exercising three times a week should include a variety of nutrient-dense foods that provide energy, support muscle recovery, and meet your overall nutritional needs. Below is a **general example of a balanced meal plan** for active individuals. Adjust portion sizes as needed based on your body weight, activity level, and goals (e.g., weight maintenance, muscle building, or weight loss). \n\n### **Daily Overview**\n- **Macronutrient Balance:** Carbohydrates (50-60%), Protein (15-25%), Fats (20-25%).  \n- **Hydration:** Drink plenty of water throughout the day, including during and after workouts.  \n- **Timing Tips:** Aim to eat every 3-4 hours to maintain energy levels and include a pre- and post-workout snack.\n\n---\n\n### **Sample 1-Day Meal Plan**\n###

## 5. Cleanup 🧹
If you'd like to remove your agent from the service once finished, you can do so below. (In production, you might keep your agent around for stateful experiences!)

In [12]:
def cleanup(agent):
    if agent:
        try:
            agents_client.delete_agent(agent.id)  # ✅
            print(f"🗑️ Deleted health advisor agent: {agent.name}")
        except Exception as e:
            print(f"❌ Error cleaning up agent: {e}")
    else:
        print("No agent to clean up.")

cleanup(health_advisor)

🗑️ Deleted health advisor agent: fun-fit-health-advisor


# Congratulations! 🏆
You've successfully built a **Fun & Fit Health Advisor** that can:
1. **Respond** to basic health and fitness questions.
2. **Use disclaimers** to encourage safe, professional consultation.
3. **Provide** general diet and wellness information.
4. **Use** the synergy of **Azure AI Foundry** modules to power the conversation.

## Next Steps
- Explore adding more advanced tools (like **FileSearchTool** or **CodeInterpreterTool**) to provide more specialized info.
- Evaluate your AI's performance with **azure-ai-evaluation**!
- Add **OpenTelemetry** or Azure Monitor for deeper insights.
- Incorporate **function calling** if you want to handle things like advanced calculation or direct data analysis.

Happy (healthy) coding! 💪